<a href="https://colab.research.google.com/github/kao2tt/AI-Trader/blob/main/%E5%8F%B0%E7%81%A3%E8%82%A1%E7%A5%A8%E7%88%AC%E8%9F%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import yfinance as yf
import pandas as pd
from datetime import datetime
import os

def fetch_taiwan_stock(stock_id, start_date, end_date):
    """
    從 Yahoo Finance 下載台灣股票數據。

    Args:
        stock_id (str): 股票代碼 (例如 "2330" 或 "2330.TW")
        start_date (str): 開始日期 (格式 "YYYY-MM-DD")
        end_date (str): 結束日期 (格式 "YYYY-MM-DD")

    Returns:
        pandas.DataFrame: 包含股價資訊的 DataFrame
    """
    # 處理股票代碼後綴
    # Yahoo Finance 規則: 上市股票為 .TW (如 2330.TW), 上櫃為 .TWO (如 8069.TWO)
    # 如果使用者沒有輸入後綴，預設先加上 .TW 嘗試下載
    ticker = stock_id.upper()
    if not (ticker.endswith('.TW') or ticker.endswith('.TWO')):
        print(f"⚠️ 偵測到未輸入後綴，將預設使用上市代碼 (.TW): {ticker}.TW")
        ticker = f"{ticker}.TW"

    print(f"正在下載 {ticker} 的資料，從 {start_date} 到 {end_date} ...")

    try:
        # 下載數據 (auto_adjust=True 會自動修正除權息後的價格)
        df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False, progress=False)

        if df.empty:
            print(f"❌ 找不到資料。請確認股票代碼 '{ticker}' 是否正確，或該日期區間無交易資料。")
            # 嘗試提示使用者是否是上櫃股票
            if ticker.endswith('.TW'):
                print(f"💡 提示: 如果這是上櫃股票，請嘗試輸入 '{stock_id}.TWO'")
            return None

        print(f"✅ 下載成功！共取得 {len(df)} 筆交易資料。")
        return df

    except Exception as e:
        print(f"❌ 下載過程中發生錯誤: {e}")
        return None

def save_to_csv(df, stock_id):
    """
    將 DataFrame 儲存為 CSV 檔案。
    """
    if df is None:
        return

    # 建立檔名，包含時間戳記避免覆蓋
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # 移除代碼中的 .TW/.TWO 以保持檔名簡潔
    clean_id = stock_id.replace('.TW', '').replace('.TWO', '')
    filename = f"{clean_id}_stock_data_{timestamp}.csv"

    try:
        # 使用 utf-8-sig 編碼，確保 Excel 開啟時中文不會亂碼
        df.to_csv(filename, encoding='utf-8-sig')
        print(f"💾 檔案已儲存至: {os.path.abspath(filename)}")
    except Exception as e:
        print(f"❌ 儲存檔案失敗: {e}")

def main():
    print("=== 台灣股票歷史資料下載器 (Yfinance) ===")
    print("請依序輸入所需資訊。")

    while True:
        # 1. 輸入股票代碼
        stock_id = input("\n請輸入股票代碼 (例如 2330 或 0050): ").strip()
        if not stock_id:
            print("股票代碼不能為空！")
            continue

        # 2. 輸入日期
        # 預設今年年初到今天
        default_start = f"{datetime.now().year}-01-01"
        default_end = datetime.now().strftime("%Y-%m-%d")

        start_date = input(f"請輸入開始日期 (YYYY-MM-DD) [預設: {default_start}]: ").strip()
        if not start_date:
            start_date = default_start

        end_date = input(f"請輸入結束日期 (YYYY-MM-DD) [預設: {default_end}]: ").strip()
        if not end_date:
            end_date = default_end

        # 3. 執行下載
        df = fetch_taiwan_stock(stock_id, start_date, end_date)

        # 4. 執行儲存
        if df is not None:
            save_to_csv(df, stock_id)

        # 詢問是否繼續
        cont = input("\n是否繼續查詢其他股票？(y/n): ").lower()
        if cont != 'y':
            print("程式結束。")
            break

if __name__ == "__main__":
    # 檢查必要的套件是否已安裝
    try:
        import yfinance
        import pandas
        main()
    except ImportError as e:
        print("❌ 缺少必要的 Python 套件。")
        print(f"錯誤訊息: {e}")
        print("請在終端機執行以下指令安裝：")
        print("pip install yfinance pandas")

=== 台灣股票歷史資料下載器 (Yfinance) ===
請依序輸入所需資訊。

請輸入股票代碼 (例如 2330 或 0050): 2882
請輸入開始日期 (YYYY-MM-DD) [預設: 2025-01-01]: 
請輸入結束日期 (YYYY-MM-DD) [預設: 2025-11-19]: 
⚠️ 偵測到未輸入後綴，將預設使用上市代碼 (.TW): 2882.TW
正在下載 2882.TW 的資料，從 2025-01-01 到 2025-11-19 ...
✅ 下載成功！共取得 212 筆交易資料。
💾 檔案已儲存至: /content/2882_stock_data_20251119_113734.csv

是否繼續查詢其他股票？(y/n): n
程式結束。
